In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

In [ ]:
class PatchEmbed(nn.Module):
    def __init__(self,
                 img_size=224,
                 patch_size=16,
                 in_chans=3,
                 embed_dim=768):
        super().__init__()
        self.img_size = img_size
        self.patch_size = patch_size
        self.in_chans = in_chans
        self.embed_dim = embed_dim
        self.num_patches = (img_size // patch_size) ** 2

        self.proj = nn.Conv2d(
            in_channels = in_chans,
            out_channels = embed_dim,
            kernel_size = patch_size,
            stride = patch_size,
            bias = bias,
        )


    def forward(self, x):
        x = self.proj(x)
        x = x.flatten(2).transpose(1, 2)
        return x

In [ ]:
class SelfAttentionEncoder(nn.Module):
    def __init__(self,
                 embed_dim=768,
                 num_heads=12,
                 attn_p=0,
                 proj_p=0,
                 flash_attn=True):
        super().__init__()
        self.num_heads = num_heads
        self.head_dim = int(embed_dim // num_heads)
        self.scale = self.head_dim ** -0.5
        self.flash_attn = flash_attn
        self.q = nn.Linear(embed_dim, embed_dim)
        self.k = nn.Linear(embed_dim, embed_dim)
        self.v = nn.Linear(embed_dim, embed_dim)
        self.attn_p = attn_p
        self.attn_drop = nn.Dropout(attn_p)
        self.proj = nn.Linear(embed_dim, embed_dim)
        self.proj_drop = nn.Dropout(proj_p)

    def forward(self, x):
        batch_size, seq_len, embed_dim = x.shape
        q = self.q(x).reshape(batch_size, seq_len, self.num_heads, self.head_dim).tranpose(1, 2)
        k = self.k(x).reshape(batch_size, seq_len, self.num_heads, self.head_dim).tranpose(1, 2)
        v = self.v(x).reshape(batch_size, seq_len, self.num_heads, self.head_dim).tranpose(1, 2)
        if self.flash_attn:
            y = F.scaled_dot_product_attention(q, k, v, dropout_p=self.attn_p if self.training else 0.0)
        else:
            attn = (q @ k.tranpose(-2, -1)) * self.scale
            attn = attn.softmax(dim=-1)
            attn = self.attn_drop(attn)
            x = attn @ v
        x = x.tranpose(1, 2).reshape(batch_size, seq_len, embed_dim)
        x = self.proj(x)
        x = self.proj_drop(x)
        return x

In [ ]:
class MLP(nn.Module):
    def __init__(self,
                 in_features,
                 mlp_ratio,
                 mlp_p=0):
        super().__init__()
        self.fl1 = nn.Linear(in_features, int(in_features * mlp_ratio))
        self.act = nn.GELU()
        self.drop1 = nn.Dropout(mlp_p)
        self.fl2 = nn.Linear(int(in_features * mlp_ratio), in_features)
        self.drop2 = nn.Dropout(mlp_p)

    def forward(self, x):
        x = self.fl1(x)
        x = self.act(x)
        x = self.drop1(x)
        x = self.fl2(x)
        x = self.drop2(x)
        return x

In [ ]:
class EncoderBlock(nn.Module):
    def __init__(self,
                 flash_attn=True,
                 embed_dim=768,
                 num_heads=12,
                 mlp_ratio=4,
                 proj_p=0.0,
                 attn_p=0.0,
                 mlp_p=0):
        super().__init__()
        self.norm1 = nn.LayerNorm(embed_dim)
        self.attn = SelfAttentionEncoder(embed_dim=embed_dim,
                                         num_heads=num_heads,
                                         attn_p=attn_p,
                                         proj_p=proj_p,
                                         flash_attn=flash_attn)
        self.norm2 = nn.LayerNorm(embed_dim)
        self.mlp = MLP(in_features=embed_dim,
                       mlp_ratio=mlp_ratio,
                       mlp_p=mlp_p)
    def forward(self, x):
        x = x + self.attn(self.norm1(x))
        x = x + self.mlp(self.norm2(x))
        return x

In [ ]:
class VisionTransformer(nn.Module):
    def __init__(self,
                 img_size=224,
                 patch_size=16,
                 in_chans=3,
                 num_classes=1000,
                 flash_attention=True,
                 embed_dim=768,
                 depth=12,
                 num_heads=12,
                 mlp_ratio=4,
                 attn_p=0.0,
                 mlp_p=0.0,
                 proj_p=0.0,
                 pos_p=0.0,
                 head_p=0.0,
                 pooling="cls", # "cls", "avg"
                 custom_weight_init=True):
        super().__init__()
        self.pooling = pooling
        assert self.pooling in ["cls", "avg"], "pooling must be either cls or avg"
        self.patch_embed = PatchEmbed(img_size=img_size,
                                      patch_size=patch_size,
                                      in_chans=in_chans,
                                      embed_dim=embed_dim)
        if pooling == "cls":
            num_tokens = self.patch_embed.num_patches + 1
        else:
            num_tokens = self.patch_embed.num_patches
        self.cls_token = nn.Parameter(torch.zeros(1, 1, embed_dim))
        self.pos_embed = nn.Parameter(torch.zeros(1, num_tokens, embed_dim))

        self.pos_drop = nn.Dropout(pos_p)

        self.blocks = nn.ModuleList(
            [
                EncoderBlock(flash_attn=flash_attn,
                             embed_dim=embed_dim,
                             num_heads=num_heads,
                             mlp_ratio=mlp_ratio,
                             proj_p=proj_p,
                             attn_p=attn_p,
                             mlp_p=mlp_p)

                for _ in range(depth)
            ]
        )
        self.norm = nn.LayerNorm(embed_dim)
        self.head_drop = nn.Dropout(head_p)
        self.head = nn.Linear(embed_dim, num_classes)

        self.apply(self._init_weights)


    def _cls_pos_embed(self, x):
        if self.pooling == "cls":
            cls_expand = self.cls_token.expand(x.shape[0], -1, -1)
            x = torch.cat([cls_expand, x], dim=1)
        x = x + self.pos_embed
        x = self.pos_drop(x)
        return x


    def _init_weights(self, module: nn.Module):
        if isinstance(module, VisionTransformer):
            module.cls_token.data = nn.init.trunc_normal_(module.cls_token.data, mean=0, std=0.02)
            module.pos_embed.data = nn.init.trunc_normal_(module.pos_embed.data, mean=0, std=0.02)

        elif isinstance(module, (nn.Linear, nn.Conv2d)):
            module.weight.data = nn.init.trunc_normal_(module.weight.data, mean=0, std=0.02)
            if module.bias is not None:
                module.bias.data.zero_()

        elif isinstance(module, nn.LayerNorm):
            module.bias.data.zero_()
            module.weight.data.fill_(1.0)


    def forward(self, x):
        x = self.patch_embed(x)
        x = self._cls_pos_embed(x)
        for block in self.blocks:
            x = block(x)
        x = self.norm(x)
        if self.pooling == "cls":
            x = x[:, 0]
        else:
            x = x.mean(dim=1)
        x = self.head_drop(x)
        x = self.head(x)
        return x


utils.py

In [ ]:
import torch
from torchvision import transforms
from torchvision.transforms import v2
from torchvision.transforms.functional import InterpolationMode
from torch.utils.data import default_collate

# set transforms for training and testing
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD = [0.229, 0.224, 0.225]

def train_transforms(image_size=(224, 224),
                     image_mean=IMAGENET_MEAN,
                     image_std=IMAGENET_STD,
                     hflip_probability=0.5,
                     interpolation=InterpolationMode.BILINEAR,
                     random_aug_magnitude=9):
    transformation_chain = []
    transformation_chain.append(v2.RandomResizedCrop(image_size, interpolation=interpolation, antialias=True))

    if hflip_probability > 0:
        transformation_chain.append(v2.RandomHorizontalFlip(p=hflip_probability))

    if random_aug_magnitude > 0:
        transformation_chain.append(v2.RandAugment(magnitude=random_aug_magnitude, interpolation=interpolation))

    transformation_chain.append(v2.PILToTensor())

    trainsformation_chain.append(v2.ToDtype(torch.float32, scale=True))

    trainsformation_chain.append(v2.Normalize(mean=image_mean, std=image_std))

    return transforms.Compose(transformation_chain)


def eval_transforms(image_size=(224, 224),
                    resize_size=(256, 256),
                    image_mean=IMAGENET_MEAN,
                    image_std=IMAGENET_STD,
                    interpolation=InterpolationMode.BILINEAR):
    transformations = transforms.Compose(
        [
            v2.Resize(resize_size, interpolation=interpolation, antialias=True),
            v2.CenterCrop(image_size),
            v2.PILToTensor(),
            v2.ToDtype(torch.float32, scale=True),
            v2.Normalize(mean=image_mean, std=image_std),
        ]
    )
    return transformations

In [ ]:
def mixup_cutmix_collate_fn(mixup_alpha=0.2,
                            cutmix_alpha=1.0,
                            num_classes=1000):
    mix_cut_transform = None
    mixup_cutmix = []

    if mixup_alpha > 0:
        mixup_cutmix.append(v2.Mixup(alpha=mixup_alpha, num_classes=num_classes))

    if cutmix_alpha > 0:
        mixup_cutmix.append(v2.CutMix(alpha=cutmix_alpha, num_classes=num_classes))

    if len(mixup_cutmix) > 0:
        mix_cut_transform = v2.RandomChoice(mixup_cutmix)

    def collate_fn(batch):
        collated = default_collate(batch)

        if mix_cut_transform is not None:
            collated = mix_cut_transform(collated)

        return collated

    return collate_fn

In [ ]:
def accuracy(output, target, topk=(1, 5)):

    with torch.inference_mode():
        maxk = max(topk)

        batch_size = target.shape[0]

        if target.ndim == 2:
            target = target.max(dim=-1)[1]

        values, pred = output.topk(maxk, dim=-1, largest=True, sorted=True)

        pred = pred.transpose(0, 1)

        correct = (pred == target)

        accs = []
        for k in topk:
            correct_k = correct[:k].flatten().sum(dtype=torch.float32)
            accs.append((correct_k / batch_size))


        if len(accs) == 1:
            return accs[0]

        else:
            return accs

train.py

In [ ]:
import os
import numpy as np
import torch
import torch.nn as nn
import argparse
from tqdm import tqdm
from torch.utils.data import DataLoader
from torchvision import datasets
from accelerate import Accelerator
from transformers import get_cosine_schedule_with_warmup

import utils
from model import VisionTransformer

import warnings
warnings.filterwarnings("ignore")

def parse_args():
    ### Parse Training Arguments ###
    parser = argparse.ArgumentParser(description="Arguments for Image Classification Training")

    ### EXPERIMENT CONFIG ###
    parser.add_argument("--experiment_name",
                        help="Name of Experiment being Launched",
                        required=True,
                        type=str)
    parser.add_argument("--wandb_run_name",
                        required=True,
                        type=str)
    parser.add_argument("--path_to_data",
                        help="Path to ImageNet root folder which should contain \train and \validation folders",
                        required=True,
                        type=str)
    parser.add_argument("--working_directory",
                        help="Working Directory where checkpoints and logs are stored, inside a \
                        folder labeled by the experiment name",
                        required=True,
                        type=str)


    ### TRAINING CONFIG ###
    parser.add_argument("--num_classes",
                        help="Number of output classes for Image Classification",
                        default=1000,
                        type=int)
    parser.add_argument("--epochs",
                        help="Number of Epochs to Train",
                        default=300,
                        type=int)
    parser.add_argument("--warmup_epochs",
                        help="Number of Epochs to Train",
                        default=30,
                        type=int)
    parser.add_argument("--save_checkpoint_interval",
                        help="After how many epochs to save model checkpoints",
                        default=10,
                        type=int)
    parser.add_argument("--per_gpu_batch_size",
                        help="Effective batch size. If split_batches is false, batch size is \
                            multiplied by number of GPUs utilized ",
                        default=256,
                        type=int)
    parser.add_argument("--gradient_accumulation_steps",
                        help="Number of Gradient Accumulation Steps for Training",
                        default=1,
                        type=int)
    parser.add_argument("--learning_rate",
                        help="Max Learning rate for cosine scheduler",
                        default=0.003,
                        type=float)
    parser.add_argument("--weight_decay",
                        help="Weight decay for optimizer",
                        default=0.1,
                        type=float)
    parser.add_argument("--random_aug_magnitude",
                        help="Magnitude of random augments, if 0 the no random augment will be applied",
                        default=9,
                        type=int)
    parser.add_argument("--mixup_alpha",
                        help="Alpha parameter for Beta distribution from which mixup lambda is sampled",
                        default=1.0,
                        type=float)
    parser.add_argument("--cutmix_alpha",
                        help="Alpha parameter for Beta distribution from which cutmix lambda is samples",
                        default=1.0,
                        type=float)
    parser.add_argument("--label_smoothing",
                        help="smooths labels when computing loss, mix between ground truth and uniform",
                        default=0,
                        type=float)
    parser.add_argument("--custom_weight_init",
                        help="Do you want to initialize the model with truncated normal layers?",
                        default=False,
                        action=argparse.BooleanOptionalAction)
    parser.add_argument("--bias_weight_decay",
                        help="Apply weight decay to bias",
                        default=False,
                        action=argparse.BooleanOptionalAction)
    parser.add_argument("--norm_weight_decay",
                        help="Apply weight decay to normalization weight and bias",
                        default=False,
                        action=argparse.BooleanOptionalAction)
    parser.add_argument("--max_grad_norm",
                        help="Maximum norm for gradient clipping",
                        default=1.0,
                        type=float)

    ### DATALOADER CONFIG ###
    parser.add_argument("--img_size",
                        help="Width and Height of Images passed to model",
                        default=224,
                        type=int)
    parser.add_argument("--num_workers",
                        help="Number of workers for DataLoader",
                        default=32,
                        type=int)


    ### EXTRA CONFIGS ###
    parser.add_argument("--log_wandb",
                        action=argparse.BooleanOptionalAction,
                        default=False)

    parser.add_argument("--resume_from_checkpoint",
                        help="Checkpoint folder for model to resume training from, inside the experiment folder",
                        default=None,
                        type=str)

    args = parser.parse_args()

    return args

### Grab Arguments ###
args = parse_args()

### Init Accelerator ###
path_to_experiment = os.path.join(args.working_directory, args.experiment_name)
accelerator = Accelerator(project_dir=path_to_experiment,
                          gradient_accumulation_steps=args.gradient_accumulation_steps,
                          log_with="wandb" if args.log_wandb else None)

### Weights and Biases Logger ###
if args.log_wandb:
    experiment_config = {"epochs": args.epochs,
                        "effective_batch_size": args.per_gpu_batch_size*accelerator.num_processes,
                        "learning_rate": args.learning_rate,
                        "warmup_epochs": args.warmup_epochs,
                        "rand_augment": args.random_aug_magnitude,
                        "cutmix_alpha": args.cutmix_alpha,
                        "mixup_alpha": args.mixup_alpha,
                        "custom_weight_init": args.custom_weight_init}

    accelerator.init_trackers(args.experiment_name, config=experiment_config, init_kwargs={"wandb": {"name": args.wandb_run_name}})

### Load Model ###
model = VisionTransformer(img_size=args.img_size,
                          num_classes=args.num_classes,
                          custom_weight_init=args.custom_weight_init)

### Set Transforms for Training and Testing ###
train_transforms = utils.train_transformations(image_size=(args.img_size, args.img_size),
                                               random_aug_magnitude=args.random_aug_magnitude)
test_transform = utils.eval_transformations()

### Load Dataset ###
path_to_train_data = os.path.join(args.path_to_data, "train")
path_to_valid_data = os.path.join(args.path_to_data, "validation")
trainset = datasets.ImageFolder(path_to_train_data, transform=train_transforms)
testset = datasets.ImageFolder(path_to_valid_data, transform=test_transform)


### Prep DataLoader with Custom Collate Function (No need on Validation only for Training) ###
mini_batchsize = args.per_gpu_batch_size // args.gradient_accumulation_steps
mixup_cutmix_collate_fn = utils.mixup_cutmix_collate_fn(mixup_alpha=args.mixup_alpha,
                                                        cutmix_alpha=args.cutmix_alpha,
                                                        num_classes=args.num_classes)
trainloader = DataLoader(trainset,
                         batch_size=mini_batchsize,
                         shuffle=True,
                         collate_fn=mixup_cutmix_collate_fn,
                         num_workers=args.num_workers,
                         pin_memory=True)

testloader = DataLoader(testset,
                        batch_size=mini_batchsize,
                        shuffle=True,
                        num_workers=args.num_workers,
                        pin_memory=True)

### Define Loss Function ###
loss_fn = nn.CrossEntropyLoss(label_smoothing=args.label_smoothing)

### Define Optimizer (And seperate out weight decay and no weight decay parameters) ###
if (not args.bias_weight_decay) or (not args.norm_weight_decay):
    accelerator.print("Disabling Weight Decay on Some Parameters")
    weight_decay_params = []
    no_weight_decay_params = []
    for name, param in model.named_parameters():

        if param.requires_grad:

            ### Dont have Weight decay on any bias parameter (including norm) ###
            if "bias" in name and not args.bias_weight_decay:
                no_weight_decay_params.append(param)

            ### Dont have Weight Decay on any Norm scales params (weights) ###
            elif "bn" in name and not args.norm_weight_decay:
                no_weight_decay_params.append(param)

            else:
                weight_decay_params.append(param)

    optimizer_group = [
        {"params": weight_decay_params, "weight_decay": args.weight_decay},
        {"params": no_weight_decay_params, "weight_decay": 0.0}
    ]
    optimizer = torch.optim.AdamW(optimizer_group, lr=args.learning_rate)

else:
    optimizer = torch.optim.AdamW(model.parameters(), lr=args.learning_rate, weight_decay=args.weight_decay)

### Define Scheduler (Compute number of training steps from epochs and adjust for num_gpus) ###
num_training_steps = len(trainloader) * args.epochs // args.gradient_accumulation_steps
num_warmup_steps = len(trainloader) * args.warmup_epochs // args.gradient_accumulation_steps
scheduler = get_cosine_schedule_with_warmup(optimizer=optimizer,
                                            num_warmup_steps=num_warmup_steps,
                                            num_training_steps=num_training_steps)

### Prepare Everything ###
model, optimizer, trainloader, testloader, scheduler = accelerator.prepare(
    model, optimizer, trainloader, testloader, scheduler
)
accelerator.register_for_checkpointing(scheduler)

### Check if we are resuming from checkpoint ###
if args.resume_from_checkpoint is not None:
    accelerator.print(f"Resuming from Checkpoint: {args.resume_from_checkpoint}")
    path_to_checkpoint = os.path.join(path_to_experiment, args.resume_from_checkpoint)
    accelerator.load_state(path_to_checkpoint)
    starting_checkpoint = int(args.resume_from_checkpoint.split("_")[-1])
else:
    starting_checkpoint = 0

for epoch in range(starting_checkpoint, args.epochs):

    accelerator.print(f"Training Epoch {epoch}")

    ### Storage for Everything ###
    train_loss = []
    test_loss = []
    train_top1_acc = []
    train_top5_acc = []
    test_top1_acc = []
    test_top5_acc = []

    accumulated_loss = 0
    accumulated_top1_accuracy = 0
    accumulated_top5_accuracy = 0

    ### Training Progress Bar ###
    progress_bar = tqdm(range(len(trainloader)//args.gradient_accumulation_steps),
                        disable=not accelerator.is_local_main_process)

    model.train()
    for images, targets in trainloader:

        ### Move Data to Correct GPU ###
        images, targets = images.to(accelerator.device), targets.to(accelerator.device)

        with accelerator.accumulate(model):

            ### Pass Through Model ###
            pred = model(images)

            ### Compute and Store Loss ##
            loss = loss_fn(pred, targets)
            accumulated_loss += loss / args.gradient_accumulation_steps

            ### Compute and Store Accuracy ###
            # predicted = pred.argmax(axis=1)
            # targets = targets.argmax(axis=1)
            top1_acc, top5_acc = utils.accuracy(pred, targets)
            accumulated_top1_accuracy += top1_acc / args.gradient_accumulation_steps
            accumulated_top5_accuracy += top5_acc / args.gradient_accumulation_steps

            ### Compute Gradients ###
            accelerator.backward(loss)

            ### Clip Gradients ###
            if accelerator.sync_gradients:
                accelerator.clip_grad_norm_(model.parameters(), args.max_grad_norm)

            ### Update Model ###
            optimizer.step()
            optimizer.zero_grad(set_to_none=True)
            scheduler.step()


        ### Only when GPUs are being synchronized (When all the grad accumulation is done) store metrics ###
        if accelerator.sync_gradients:

            ### Gather Metrics Across GPUs ###
            loss_gathered = accelerator.gather_for_metrics(accumulated_loss)
            top1_accuracy_gathered = accelerator.gather_for_metrics(accumulated_top1_accuracy)
            top5_accuracy_gathered = accelerator.gather_for_metrics(accumulated_top5_accuracy)

            ### Store Current Iteration Error ###
            train_loss.append(torch.mean(loss_gathered).item())
            train_top1_acc.append(torch.mean(top1_accuracy_gathered).item())
            train_top5_acc.append(torch.mean(top5_accuracy_gathered).item())

            ### Reset Accumulated for next Accumulation ###
            accumulated_loss, accumulated_top1_accuracy, accumulated_top5_accuracy = 0, 0, 0

            ### Iterate Progress Bar ###
            progress_bar.update(1)


    model.eval()
    for images, targets in tqdm(testloader, disable=not accelerator.is_local_main_process):
        images, targets = images.to(accelerator.device), targets.to(accelerator.device)
        with torch.no_grad():
            pred = model(images)

        ### Compute Loss ###
        loss = loss_fn(pred, targets)

        ### Computed Accuracy ###
        # predicted = pred.argmax(axis=1)
        top1_acc, top5_acc = utils.accuracy(pred, targets)

        ### Gather across GPUs ###
        loss_gathered = accelerator.gather_for_metrics(loss)
        top1_accuracy_gathered = accelerator.gather_for_metrics(top1_acc)
        top5_accuracy_gathered = accelerator.gather_for_metrics(top5_acc)

        ### Store Current Iteration Error ###
        test_loss.append(torch.mean(loss_gathered).item())
        test_top1_acc.append(torch.mean(top1_accuracy_gathered).item())
        test_top5_acc.append(torch.mean(top5_accuracy_gathered).item())

    epoch_train_loss = np.mean(train_loss)
    epoch_test_loss = np.mean(test_loss)
    epoch_train_top1_acc = np.mean(train_top1_acc)
    epoch_train_top5_acc = np.mean(train_top5_acc)
    epoch_test_top1_acc = np.mean(test_top1_acc)
    epoch_test_top5_acc = np.mean(test_top5_acc)

    accelerator.print("Training Top 1 Accuracy:", round(epoch_train_top1_acc, 3), \
                      "Training Top 5 Accuracy:", round(epoch_train_top5_acc, 3), \
                      "Training Loss:", round(epoch_train_loss,3))

    accelerator.print("Testing Top 1 Accuracy:", round(epoch_test_top1_acc,3), \
                      "Testing Top 5 Accuracy:", round(epoch_test_top5_acc,3), \
                      "Testing Loss:", round(epoch_test_loss,3))

    ### Log with Weights and Biases ###
    if args.log_wandb:
        accelerator.log({"training_loss": epoch_train_loss,
                        "testing_loss": epoch_test_loss,
                        "training_top1_acc": epoch_train_top1_acc,
                        "training_top5_acc": epoch_train_top5_acc,
                        "testing_top1_acc": epoch_test_top1_acc,
                        "testing_top5_acc": epoch_test_top5_acc,
                        "learning_rate": scheduler.get_last_lr()[0]}, step=epoch)

    ### Checkpoint Model ###
    if epoch % args.save_checkpoint_interval == 0:
        path_to_checkpoint = os.path.join(path_to_experiment, f"checkpoint_{epoch}")
        accelerator.save_state(output_dir=path_to_checkpoint)

### End Training for Trackers to Exit ###
accelerator.end_training()

In [ ]:
if __name__ == "__main__":
    rand = torch.randn(4, 3, 224, 224)
    pe = PatchEmbed(rand)